In [ ]:
# ==============================================================================
# STEP 1: SETUP & LOADING (Wave 8 - 2019 Single File)
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

sns.set_theme(style="whitegrid")
file_2019 = 'wave-8-shocks.dta' 
df_19 = pd.read_stata(file_2019, convert_categoricals=False)

# ==============================================================================
# STEP 2: CLEANING & TRANSFORMATION
# ==============================================================================
def clean_wave8(df):
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 1. จัดการข้อมูลเงิน (v31105a, v31105b, v31106a)
    money_cols = ['v31105a', 'v31105b', 'v31106a']
    for col in money_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('none', '0'), errors='coerce').fillna(0)
    
    df['impact_3way'] = df['v31105a'] + df['v31105b'] + df['v31106a']
    df['impact_2way'] = df['v31105a'] + df['v31106a']
    
    # 2. Shock Grouping
    shock_mapping = {10:'agricultural', 11:'agricultural', 63:'agricultural', 55:'agricultural',
                     1:'demographic', 2:'demographic', 3:'demographic', 24:'demographic',
                     5:'economics', 6:'economics', 18:'economics', 21:'economics', 22:'economics', 62:'economics',
                     8:'social', 70:'social', 77:'economics'}
    df['shocks_Group'] = df['shocks__id'].map(shock_mapping).fillna('others')
    
    # 3. Coping Rename
    coping_map = {'v31108a__1':'coping_savings', 'v31108a__2':'coping_insurance', 
                  'v31108a__3':'coping_informal_borrow', 'v31108a__4':'coping_formal_borrow', 
                  'v31108a__5':'coping_assets', 'v31108a__7':'coping_gov_help'}
    df = df.rename(columns=coping_map)
    return df

df_19 = clean_wave8(df_19)

output_file = 'shocks_2019_cleaned_final.csv'
df_19.to_csv(output_file, index=False)

In [ ]:
# ==============================================================================
# STEP 3: MASTER 8 GRAPHS ANALYSIS
# ==============================================================================
def run_8_graphs(df, year):
    output_dir = f"Descriptive_Graph_{year}"
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    # Standardizing Recovery (Match Wave 1)
    def map_rec(m):
        if pd.isna(m): return np.nan
        if m < 12: return "less than 1 year"
        if m == 12: return "1 year"
        if 12 < m < 90: return "more than 1 year, but recovered"
        return "not yet recovered" if m >= 90 else np.nan
    
    df['recovery_std'] = df['v31112a'].apply(map_rec)
    df['cons_label'] = df['v31111'].map({1: "Yes (Reduced)", 2: "No (Did not reduce)"})
    coping_vars = [c for c in df.columns if c.startswith('coping_')]

    # G1-G8 Logic
    plots = [
        ('G1_Frequency', lambda: sns.countplot(data=df, x='shocks_Group', palette='viridis')),
        ('G2_Impact_3way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_3way', estimator=np.mean)),
        ('G3_Loss_2way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_2way', estimator=np.mean)),
        ('G4_CopingUsage', lambda: df[coping_vars].sum().sort_values().plot(kind='barh')),
        ('G5_CopingIntensity', lambda: sns.countplot(data=df, x=df[coping_vars].sum(axis=1))),
        ('G6_RecoveryMonths', lambda: sns.histplot(data=df[df['v31112a']<90], x='v31112a', bins=20, kde=True)),
        ('G7_RecoveryStd', lambda: sns.countplot(data=df, x='recovery_std', order=["less than 1 year", "1 year", "more than 1 year, but recovered", "not yet recovered"])),
        ('G8_Consumption', lambda: sns.countplot(data=df[df['cons_label'].notnull()], x='cons_label'))
    ]

    for name, func in plots:
        plt.figure(figsize=(10, 6)); func(); plt.title(f"{name} ({year})")
        if any(x in name for x in ['G1','G2','G3']): plt.xticks(rotation=45)
        plt.savefig(f"{output_dir}/{name}.png", dpi=300, bbox_inches='tight'); plt.show()

run_8_graphs(df_19, 2019)